# 23 — ShallowMLP + ET/CNN/MLP Diversity Check & Ensemble

**目的**: 浅いMLPを追加素材として多様性確認し、ET+CNN+MLPアンサンブルを評価。

**前処理**: SNV+SG1(41,3,1) — 全モデル共通・固定。多様性はモデル種のみから出す。

**Step 1**: ShallowMLP 単体 (3-seed avg) の性能確認
**Step 2**: ET / CNN / MLP の OOF 予測相関・残差相関で多様性判定
**Step 3**: 基準①②を満たした場合のみ ET+CNN+MLP アンサンブル構築

判定基準: MLP の CNN との相関 ≥ 0.97 → 加える価値薄い。< 0.97 → 加える価値あり。

In [ ]:
import sys, os, copy
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import GroupKFold
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from src.utils import load_data, parse_spectra, get_groups, make_submission

SEED   = 42
SEEDS  = [42, 123, 456]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLIP_T = 200.0
ET_KW  = dict(n_estimators=300, max_features=0.3, random_state=SEED, n_jobs=-1)

train_df, test_df = load_data()
train_meta, y_s, X_raw, wn = parse_spectra(train_df)
test_meta,  _,   X_test_raw, _ = parse_spectra(test_df)
y      = y_s.values.astype(float)
groups = get_groups(train_meta)
SPLITS = list(GroupKFold(n_splits=5).split(X_raw, y, groups))

# Precompute SNV+SG1 for all data (row-wise SNV + fixed SG1, no fold-specific fit)
def snv_sg1(R):
    A = R.copy().astype(np.float64)
    A = (A - A.mean(1, keepdims=True)) / (A.std(1, keepdims=True) + 1e-8)
    A = savgol_filter(A, window_length=41, polyorder=3, deriv=1, axis=1)
    return A.astype(np.float32)

X_pp    = snv_sg1(X_raw)
X_pp_te = snv_sg1(X_test_raw)
N_IN    = X_pp.shape[1]  # 1555

def rmse_all(yt, yp): return float(np.sqrt(np.mean((np.asarray(yt)-np.asarray(yp))**2)))
def rmse_le(yt, yp, T=170.0):
    yt, yp = np.asarray(yt), np.asarray(yp)
    m = yt <= T
    return float(np.sqrt(np.mean((yt[m]-yp[m])**2))) if m.sum()>0 else np.nan

print(f'Device: {DEVICE}')
print(f'Train: {X_raw.shape}  Test: {X_test_raw.shape}')
print(f'X_pp: {X_pp.shape}  N_IN={N_IN}')
print(f'Folds: {len(SPLITS)}  Seeds: {SEEDS}')

In [ ]:
# ===== ImprovedCNN1D (same as nb18) =====
class ImprovedCNN1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.block12 = nn.Sequential(
            nn.Conv1d(1,  8,  kernel_size=15, padding=7), nn.BatchNorm1d(8),  nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(8,  16, kernel_size=9,  padding=4), nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(2),
        )
        self.conv3     = nn.Sequential(nn.Conv1d(16, 32, kernel_size=5, padding=2), nn.BatchNorm1d(32))
        self.shortcut3 = nn.Conv1d(16, 32, kernel_size=1)
        self.pool = nn.AdaptiveAvgPool1d(16)
        self.fc = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(32*16, 32), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(32, 1),
        )
    def forward(self, x):
        h = self.block12(x.unsqueeze(1))
        h = torch.relu(self.conv3(h) + self.shortcut3(h))
        return self.fc(self.pool(h).view(x.size(0), -1)).squeeze(1)


# ===== ShallowMLP (specified in task) =====
class ShallowMLP(nn.Module):
    def __init__(self, n_in, hidden=128, p=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(p),
            nn.Linear(hidden, 32),   nn.BatchNorm1d(32),    nn.ReLU(), nn.Dropout(p),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(1)


n_cnn = sum(p.numel() for p in ImprovedCNN1D().parameters())
n_mlp = sum(p.numel() for p in ShallowMLP(N_IN).parameters())
print(f'ImprovedCNN1D : {n_cnn:,} params')
print(f'ShallowMLP    : {n_mlp:,} params  (hidden=128, p=0.4)')
if n_mlp > 500_000:
    print('WARNING: MLP params > 500k — consider reducing hidden')
else:
    print('MLP param count OK')

In [ ]:
# ===== Training functions (shared logic) =====

def _train_nn(model, loader, Xva_t, yva_t, opt, sched, crit,
              n_epochs=100, patience=20):
    best_val, best_state, best_preds = float('inf'), None, None
    no_improve = 0; stop_ep = n_epochs
    for epoch in range(n_epochs):
        model.train()
        for xb, yb in loader:
            loss = crit(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            vp = model(Xva_t)
            vl = crit(vp, yva_t).item()
        if vl < best_val:
            best_val = vl
            best_state = copy.deepcopy(model.state_dict())
            best_preds = vp.cpu().numpy()
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            stop_ep = epoch + 1; break
    model.load_state_dict(best_state)
    return model, best_preds, stop_ep


def train_cnn(Xtr_s, ytr, Xva_s, yva, seed, n_epochs=100, batch=32, lr=1e-3, patience=20):
    torch.manual_seed(seed); np.random.seed(seed)
    Xtr_t = torch.from_numpy(Xtr_s).to(DEVICE)
    ytr_t = torch.from_numpy(ytr.astype(np.float32)).to(DEVICE)
    Xva_t = torch.from_numpy(Xva_s).to(DEVICE)
    yva_t = torch.from_numpy(yva.astype(np.float32)).to(DEVICE)
    loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch, shuffle=True)
    model = ImprovedCNN1D().to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    return _train_nn(model, loader, Xva_t, yva_t, opt, sched,
                     nn.HuberLoss(delta=10.0), n_epochs, patience)


def train_mlp(Xtr_s, ytr, Xva_s, yva, seed, n_in,
              n_epochs=100, batch=32, lr=1e-3, wd=1e-4, patience=20):
    torch.manual_seed(seed); np.random.seed(seed)
    Xtr_t = torch.from_numpy(Xtr_s).to(DEVICE)
    ytr_t = torch.from_numpy(ytr.astype(np.float32)).to(DEVICE)
    Xva_t = torch.from_numpy(Xva_s).to(DEVICE)
    yva_t = torch.from_numpy(yva.astype(np.float32)).to(DEVICE)
    loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch, shuffle=True)
    model = ShallowMLP(n_in).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    return _train_nn(model, loader, Xva_t, yva_t, opt, sched,
                     nn.MSELoss(), n_epochs, patience)


print('Training functions defined.')

## Combined GroupKFold CV — ET / CNN(3-seed) / MLP(3-seed)

同一 fold・同一 StandardScaler を全モデルで共有。
ET は StandardScaler 不要（木モデルのため）。
Fold3 の高誤差は構造的問題。改善目標にしない。

In [ ]:
print('=== Combined GroupKFold CV ===')
print('ET + CNN(3-seed) + MLP(3-seed) on same folds / same scaler')
print()

oof_et  = np.zeros(len(y))
oof_cnn = np.zeros(len(y))
oof_mlp = np.zeros(len(y))

cnn_stops_per_fold = []
mlp_stops_per_fold = []

for fi, (tr, va) in enumerate(SPLITS):
    Xtr_pp, Xva_pp = X_pp[tr], X_pp[va]
    ytr, yva = y[tr], y[va]

    # StandardScaler — fit on train, shared by CNN + MLP
    sc = StandardScaler()
    Xtr_s = sc.fit_transform(Xtr_pp).astype(np.float32)
    Xva_s = sc.transform(Xva_pp).astype(np.float32)

    # ----- ET -----
    et_m = ExtraTreesRegressor(**ET_KW)
    et_m.fit(Xtr_pp, ytr)
    oof_et[va] = et_m.predict(Xva_pp)

    # ----- CNN 3-seed -----
    cnn_sp, cnn_st = [], []
    for seed in SEEDS:
        _, pred, stop = train_cnn(Xtr_s, ytr, Xva_s, yva, seed)
        cnn_sp.append(pred); cnn_st.append(stop)
    oof_cnn[va] = np.mean(cnn_sp, 0)
    cnn_stops_per_fold.append(cnn_st)

    # ----- MLP 3-seed -----
    mlp_sp, mlp_st = [], []
    for seed in SEEDS:
        _, pred, stop = train_mlp(Xtr_s, ytr, Xva_s, yva, seed, N_IN)
        mlp_sp.append(pred); mlp_st.append(stop)
    oof_mlp[va] = np.mean(mlp_sp, 0)
    mlp_stops_per_fold.append(mlp_st)

    print(f'Fold {fi+1}: ET={rmse_le(yva, oof_et[va]):.2f}%  '
          f'CNN={rmse_le(yva, oof_cnn[va]):.2f}%  '
          f'MLP={rmse_le(yva, oof_mlp[va]):.2f}%  '
          f'cnn_stops={cnn_st}  mlp_stops={mlp_st}')

cnn_avg_stop = int(round(np.mean(cnn_stops_per_fold)))
mlp_avg_stop = int(round(np.mean(mlp_stops_per_fold)))
print(f'\ncnn_avg_stop={cnn_avg_stop}  mlp_avg_stop={mlp_avg_stop}')
print('CV complete.')

## Step 1: MLP 単体性能チェック

In [ ]:
# Per-model fold results from OOF
def fold_summary(oof_p, label):
    folds_le = [rmse_le(y[va], oof_p[va]) for _, (_, va) in enumerate(SPLITS)]
    folds_all = [rmse_all(y[va], oof_p[va]) for _, (_, va) in enumerate(SPLITS)]
    mean_le = float(np.mean(folds_le))
    std_no3 = float(np.std([folds_le[i] for i in [0, 1, 3, 4]]))
    print(f'{label}:')
    print(f'  Folds: {[round(x,2) for x in folds_le]}')
    print(f'  Mean={mean_le:.2f}%  Std(F1,2,4,5)={std_no3:.2f}%')
    print(f'  OOF dist: min={oof_p.min():.1f}  mean={oof_p.mean():.1f}  '
          f'max={oof_p.max():.1f}  >170:{(oof_p>170).sum()}')
    return folds_le, mean_le, std_no3

print('=== Step 1: 各モデル単体 CV サマリ ===')
print()
et_folds,  et_mean,  et_std  = fold_summary(oof_et,  'ET  (SNV+SG1+ExtraTrees)')
print()
cnn_folds, cnn_mean, cnn_std = fold_summary(oof_cnn, 'CNN (ImprovedCNN1D 3-seed)')
print()
mlp_folds, mlp_mean, mlp_std = fold_summary(oof_mlp, 'MLP (ShallowMLP 3-seed)')
print()

# Step 1 criterion
print('--- 基準① MLP 性能チェック ---')
mlp_ok = (mlp_mean < 30) and (oof_mlp.mean() > 30) and (oof_mlp.mean() < 65)
print(f'MLP mean CV = {mlp_mean:.2f}%  (基準: < 30%)')
print(f'MLP OOF mean = {oof_mlp.mean():.1f}%  (健全範囲: 30-65%)')
print(f'判定: {"OK — Step 2 へ" if mlp_ok else "NG — 素材として弱い"}'  )

## Step 2: 多様性確認（OOF予測・残差の相関）

In [ ]:
print('=== Step 2: 多様性確認 ===')
print('y<=170 のサンプルのみで相関を計算（高含水率外れ値を除外）')
print()

mask = y <= 170
oof_et_m  = oof_et[mask]
oof_cnn_m = oof_cnn[mask]
oof_mlp_m = oof_mlp[mask]
y_m       = y[mask]

# 予測の相関
r_et_cnn  = np.corrcoef(oof_et_m,  oof_cnn_m)[0, 1]
r_et_mlp  = np.corrcoef(oof_et_m,  oof_mlp_m)[0, 1]
r_cnn_mlp = np.corrcoef(oof_cnn_m, oof_mlp_m)[0, 1]

# 残差の相関（実測-予測）
res_et  = y_m - oof_et_m
res_cnn = y_m - oof_cnn_m
res_mlp = y_m - oof_mlp_m
rr_et_cnn  = np.corrcoef(res_et,  res_cnn)[0, 1]
rr_et_mlp  = np.corrcoef(res_et,  res_mlp)[0, 1]
rr_cnn_mlp = np.corrcoef(res_cnn, res_mlp)[0, 1]

print('予測相関 (r):')
print(f'  ET  vs CNN : {r_et_cnn:.4f}')
print(f'  ET  vs MLP : {r_et_mlp:.4f}')
print(f'  CNN vs MLP : {r_cnn_mlp:.4f}  <- 主判定')
print()
print('残差相関 (r_residual):')
print(f'  ET  vs CNN : {rr_et_cnn:.4f}')
print(f'  ET  vs MLP : {rr_et_mlp:.4f}')
print(f'  CNN vs MLP : {rr_cnn_mlp:.4f}  <- 主判定')
print()

THRESH = 0.97
mlp_diverse = (r_cnn_mlp < THRESH) or (rr_cnn_mlp < THRESH)
print(f'--- 基準② 多様性チェック ---')
print(f'閾値: CNN vs MLP 予測相関 < {THRESH}  または  残差相関 < {THRESH}')
print(f'CNN vs MLP 予測 r={r_cnn_mlp:.4f}  残差 r={rr_cnn_mlp:.4f}')
print(f'判定: {"OK — Step 3 へ（MLP に多様性あり）" if mlp_diverse else "NG — MLP は CNN とほぼ同一の予測、加える価値薄い"}'  )

# Scatter plot
os.makedirs('../results', exist_ok=True)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
pairs = [('ET', oof_et_m, 'CNN', oof_cnn_m, r_et_cnn),
         ('ET', oof_et_m, 'MLP', oof_mlp_m, r_et_mlp),
         ('CNN', oof_cnn_m, 'MLP', oof_mlp_m, r_cnn_mlp)]
for ax, (na, pa, nb_, pb, r) in zip(axes, pairs):
    ax.scatter(pa, pb, s=4, alpha=0.3)
    lim = max(pa.max(), pb.max()) * 1.05
    ax.plot([0, lim], [0, lim], 'r--', lw=0.8)
    ax.set_xlabel(f'{na} pred'); ax.set_ylabel(f'{nb_} pred')
    ax.set_title(f'{na} vs {nb_}  r={r:.4f}')
    ax.grid(True, alpha=0.3)
plt.suptitle('OOF prediction scatter (y<=170)', y=1.02)
plt.tight_layout()
plt.savefig('../results/nb23_diversity.png', dpi=110)
plt.close()
print('Saved: results/nb23_diversity.png')

## Test Predictions — ET / CNN / MLP

In [ ]:
print('=== Test Predictions (full train) ===')
os.makedirs('../submissions', exist_ok=True)

# ----- ET test -----
et_full = ExtraTreesRegressor(**ET_KW)
et_full.fit(X_pp, y)
te_et = np.clip(et_full.predict(X_pp_te), 0, CLIP_T)
print(f'ET  test: min={te_et.min():.1f}  mean={te_et.mean():.1f}  '
      f'max={te_et.max():.1f}  >170:{(te_et>170).sum()}')

# ----- CNN test (3-seed) -----
sc_full = StandardScaler()
Xtr_s_f = sc_full.fit_transform(X_pp).astype(np.float32)
Xte_s_f = sc_full.transform(X_pp_te).astype(np.float32)
Xtr_tf  = torch.from_numpy(Xtr_s_f).to(DEVICE)
ytr_tf  = torch.from_numpy(y.astype(np.float32)).to(DEVICE)
Xte_tf  = torch.from_numpy(Xte_s_f).to(DEVICE)

cnn_te_sp = []
for seed in SEEDS:
    torch.manual_seed(seed); np.random.seed(seed)
    model_f = ImprovedCNN1D().to(DEVICE)
    opt_f   = torch.optim.Adam(model_f.parameters(), lr=1e-3)
    sched_f = torch.optim.lr_scheduler.CosineAnnealingLR(opt_f, T_max=100)
    crit_f  = nn.HuberLoss(delta=10.0)
    loader_f = DataLoader(TensorDataset(Xtr_tf, ytr_tf), batch_size=32, shuffle=True)
    for ep in range(cnn_avg_stop):
        model_f.train()
        for xb, yb in loader_f:
            loss = crit_f(model_f(xb), yb); opt_f.zero_grad(); loss.backward(); opt_f.step()
        sched_f.step()
    model_f.eval()
    with torch.no_grad():
        cnn_te_sp.append(model_f(Xte_tf).cpu().numpy())
    print(f'  CNN seed={seed} done')
te_cnn = np.clip(np.mean(cnn_te_sp, 0), 0, CLIP_T)
print(f'CNN test: min={te_cnn.min():.1f}  mean={te_cnn.mean():.1f}  '
      f'max={te_cnn.max():.1f}  >170:{(te_cnn>170).sum()}')

# ----- MLP test (3-seed) -----
mlp_te_sp = []
for seed in SEEDS:
    torch.manual_seed(seed); np.random.seed(seed)
    model_m = ShallowMLP(N_IN).to(DEVICE)
    opt_m   = torch.optim.Adam(model_m.parameters(), lr=1e-3, weight_decay=1e-4)
    sched_m = torch.optim.lr_scheduler.CosineAnnealingLR(opt_m, T_max=100)
    crit_m  = nn.MSELoss()
    loader_m = DataLoader(TensorDataset(Xtr_tf, ytr_tf), batch_size=32, shuffle=True)
    for ep in range(mlp_avg_stop):
        model_m.train()
        for xb, yb in loader_m:
            loss = crit_m(model_m(xb), yb); opt_m.zero_grad(); loss.backward(); opt_m.step()
        sched_m.step()
    model_m.eval()
    with torch.no_grad():
        mlp_te_sp.append(model_m(Xte_tf).cpu().numpy())
    print(f'  MLP seed={seed} done')
te_mlp = np.clip(np.mean(mlp_te_sp, 0), 0, CLIP_T)
print(f'MLP test: min={te_mlp.min():.1f}  mean={te_mlp.mean():.1f}  '
      f'max={te_mlp.max():.1f}  >170:{(te_mlp>170).sum()}')

make_submission(test_meta, te_cnn, '../submissions/sub_mlp_cnn_base.csv')  # CNN再生成ベース確認用
make_submission(test_meta, te_mlp, '../submissions/sub_mlp.csv')
print('Saved: sub_mlp.csv  sub_mlp_cnn_base.csv')

## Step 3: アンサンブル構築と比較（基準①②を満たした場合）

In [ ]:
print('=== Step 3: アンサンブル構築 ===')
print()

def ens_fold_summary(oof_p, label):
    folds_le = [rmse_le(y[va], oof_p[va]) for _, (_, va) in enumerate(SPLITS)]
    std_no3  = float(np.std([folds_le[i] for i in [0, 1, 3, 4]]))
    mean_le  = float(np.mean(folds_le))
    return folds_le, mean_le, std_no3

# OOF ensembles (clip to positive for physical sanity)
oof_et_cnn     = np.clip((oof_et + oof_cnn) / 2,                   0, CLIP_T)
oof_3avg       = np.clip((oof_et + oof_cnn + oof_mlp) / 3,         0, CLIP_T)
oof_cnn_mlp    = np.clip((oof_cnn + oof_mlp) / 2,                  0, CLIP_T)

# Test ensembles
te_et_cnn  = np.clip((te_et + te_cnn) / 2,                0, CLIP_T)
te_3avg    = np.clip((te_et + te_cnn + te_mlp) / 3,       0, CLIP_T)
te_cnn_mlp = np.clip((te_cnn + te_mlp) / 2,               0, CLIP_T)

configs = [
    ('ET only',         oof_et,      et_folds,  te_et,     'N/A (LB=18.35)'),
    ('CNN only',        oof_cnn,     cnn_folds, te_cnn,    'N/A (LB=17.73)'),
    ('MLP only',        oof_mlp,     mlp_folds, te_mlp,    'sub_mlp.csv'),
    ('ET+CNN avg',      oof_et_cnn,  None,      te_et_cnn, 'LB=17.88 submitted'),
    ('ET+CNN+MLP avg',  oof_3avg,    None,      te_3avg,   'sub_et_cnn_mlp_avg.csv'),
    ('CNN+MLP avg',     oof_cnn_mlp, None,      te_cnn_mlp,'sub_cnn_mlp_avg.csv'),
]

print(f'{"Model":<20} | {"F1":>6} {"F2":>6} {"F3":>6} {"F4":>6} {"F5":>6} '
      f'| {"Mean":>7} {"Std(F1245)":>11} | {"test_mean":>9} {">170":>5} | Note')
print('-' * 105)

for label, oof_p, pre_folds, te_p, note in configs:
    if pre_folds is not None:
        folds_le = [round(x, 2) for x in pre_folds]
    else:
        folds_le = [round(rmse_le(y[va], oof_p[va]), 2) for _, (_, va) in enumerate(SPLITS)]
    mean_le = float(np.mean(folds_le))
    std_no3 = float(np.std([folds_le[i] for i in [0, 1, 3, 4]]))
    te_mean = te_p.mean()
    te_gt170 = (te_p > 170).sum()
    healthy  = (30 <= te_mean <= 65) and (te_gt170 <= 30)
    h_mark   = '' if healthy else ' ⚠dist'
    print(f'{label:<20} | {folds_le[0]:>6.2f} {folds_le[1]:>6.2f} {folds_le[2]:>6.2f} '
          f'{folds_le[3]:>6.2f} {folds_le[4]:>6.2f} '
          f'| {mean_le:>7.2f} {std_no3:>11.2f} '
          f'| {te_mean:>9.1f} {te_gt170:>5} | {note}{h_mark}')

# Save ensemble CSVs
make_submission(test_meta, te_3avg,    '../submissions/sub_et_cnn_mlp_avg.csv')
make_submission(test_meta, te_cnn_mlp, '../submissions/sub_cnn_mlp_avg.csv')
print()
print('Saved: sub_et_cnn_mlp_avg.csv  sub_cnn_mlp_avg.csv')

In [ ]:
# Final summary
print()
print('=' * 75)
print('総括')
print('=' * 75)
print()

# Step 1
print(f'基準① MLP 単体性能: mean CV={mlp_mean:.2f}%  OOF_mean={oof_mlp.mean():.1f}%')
s1_ok = (mlp_mean < 30) and (30 <= oof_mlp.mean() <= 65)
print(f'  -> {"OK" if s1_ok else "NG"}: {"ET/CNNと同水準で素材として使用可能" if s1_ok else "ET/CNNより大幅に弱い、または分布異常"}')
print()

# Step 2
print(f'基準② 多様性: CNN vs MLP 予測相関 r={r_cnn_mlp:.4f}  残差相関 r={rr_cnn_mlp:.4f}')
s2_ok = (r_cnn_mlp < THRESH) or (rr_cnn_mlp < THRESH)
print(f'  -> {"OK" if s2_ok else "NG"}: {"CNN と違う間違い方をしている。アンサンブルに加える価値あり" if s2_ok else "CNN とほぼ同一予測。アンサンブル効果薄い"}')
print()

# Step 3
print('Step 3 アンサンブル頑健性 (Std F1,2,4,5 比較):')
ref_std_val = float(np.std([et_folds[i] for i in [0,1,3,4]]))  # ET baseline
for label, oof_p in [('ET+CNN', oof_et_cnn), ('ET+CNN+MLP', oof_3avg), ('CNN+MLP', oof_cnn_mlp)]:
    folds_le = [rmse_le(y[va], oof_p[va]) for _, (_, va) in enumerate(SPLITS)]
    std_no3  = float(np.std([folds_le[i] for i in [0, 1, 3, 4]]))
    print(f'  {label:<15}: Std={std_no3:.2f}%')
print()

# Recommendation
both_ok = s1_ok and s2_ok
print('提出候補まとめ:')
print(f'  sub_cnn_ensemble.csv   LB=17.73 (現最良、CNN 3-seed)')
print(f'  sub_et_cnn_avg.csv     LB=17.88 (ET+CNN、頑健性観点で保持)')
if both_ok:
    print(f'  sub_et_cnn_mlp_avg.csv NEW: ET+CNN+MLP 3種平均 -> 提出推奨')
    print(f'  sub_mlp.csv            NEW: MLP 単体 -> 参考提出可')
else:
    print(f'  sub_et_cnn_mlp_avg.csv 基準①②未達のため提出優先度低')
    print(f'  sub_mlp.csv            MLP 単体 -> 参考提出可')
print()
print('最終判断はPublic/Private。CVは健全性確認のみ。')